In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

# Load the XML file (update the filename if needed)
xml_file = r"Pies.xml"

In [35]:
items = []
descriptions = []
attributes = []
extended = []
packages = []
interchanges = []


In [36]:
for event, elem in ET.iterparse(xml_file, events=("end",)):

    tag = elem.tag.split("}")[-1]

    if tag == "Item":

        item_row = {}

        for k, v in elem.attrib.items():
            item_row[k] = v

        for child in elem:
            ctag = child.tag.split("}")[-1]

            if ctag not in ["Descriptions", "ExtendedInformation",
                            "ProductAttributes", "Packages",
                            "PartInterchangeInfo"]:

                if child.text and child.text.strip():
                    item_row[ctag] = child.text.strip()

                for a, v in child.attrib.items():
                    item_row[f"{ctag}_{a}"] = v

        items.append(item_row)

        part = item_row.get("PartNumber")
        PartTerminologyID=item_row.get("PartTerminologyID")

        # DESCRIPTIONS
        desc = elem.find(".//{*}Descriptions")
        if desc is not None:
            for d in desc:
                descriptions.append({
                    "PartNumber": part,
                    "DescriptionCode": d.attrib.get("DescriptionCode"),
                    "Text": d.text
                })

        # ATTRIBUTES
        attrs = elem.find(".//{*}ProductAttributes")
        if attrs is not None:
            for a in attrs:
                attributes.append({
                    "PartNumber": part,
                    "PartTerminologyID": PartTerminologyID,
                    "AttributeID": a.attrib.get("AttributeID"),
                    "UOM":a.attrib.get("AttributeUOM"),
                    "Value": a.text
                })

        # EXTENDED INFO
        ext = elem.find(".//{*}ExtendedInformation")
        if ext is not None:
            for e in ext:
                extended.append({
                    "PartNumber": part,
                    "Code": e.attrib.get("EXPICode"),
                    "Value": e.text
                })

        # PACKAGES
        pkg = elem.find(".//{*}Packages")
        if pkg is not None:
            for p in pkg:
                packages.append({
                    "PartNumber": part,
                    "PackageUOM": p.findtext(".//{*}PackageUOM"),
                    "Quantity": p.findtext(".//{*}QuantityofEaches")
                })

        # INTERCHANGE
        inter = elem.find(".//{*}PartInterchangeInfo")
        if inter is not None:
            for i in inter:
                interchanges.append({
                    "PartNumber": part,
                    "BrandAAIAID": i.attrib.get("BrandAAIAID"),
                    "PartNumber_Interchange": i.findtext(".//{*}PartNumber")
                })

        elem.clear()


# Convert to DataFrames
df_items = pd.DataFrame(items)
df_desc = pd.DataFrame(descriptions)
df_attr = pd.DataFrame(attributes)
df_ext = pd.DataFrame(extended)
df_pkg = pd.DataFrame(packages)
df_inter = pd.DataFrame(interchanges)


In [37]:
df_attr

,PartNumber,PartTerminologyID,AttributeID,UOM,Value
0,6300,8840,Is Or Contains A Bulb,None,No
1,6300,8840,Washer Pump PSI,None,22 psi
2,6300,8840,Washer Pump Volume,None,120 ml per 5 seconds
3,6300,8840,AAP_UDA_129,None,N
4,6300,8840,California Proposition 65,None,N
...,...,...,...,...,...
15739,EV3232PB22,11112,3145,None,Specific
15740,EV3232PB22,11112,4217,IN,32.0
15741,EV3232PB22,11112,4218,IN,32.0
15742,EV3232PB22,11112,585,None,Black


In [24]:

# Write Excel
with pd.ExcelWriter("pies_output.xlsx") as writer:
    df_items.to_excel(writer, sheet_name="Items", index=False)
    df_desc.to_excel(writer, sheet_name="Descriptions", index=False)
    df_attr.to_excel(writer, sheet_name="Attributes", index=False)
    df_ext.to_excel(writer, sheet_name="ExtendedInfo", index=False)
    df_pkg.to_excel(writer, sheet_name="Packages", index=False)
    df_inter.to_excel(writer, sheet_name="Interchange", index=False)